# Vector-Space Negatives + CatBoost

This self-contained notebook builds a Kaggle-ready ranking classifier without importing repository code or downloading external models. It creates synthetic negatives from query-item vector similarity, trains CatBoost on vector, lexical, and categorical features, and writes score and submission files to `/kaggle/working` when running on Kaggle.


In [ ]:
from __future__ import annotations

import csv
import gc
import math
import os
import re
import time
import unicodedata
import warnings
from functools import lru_cache
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from catboost import CatBoostClassifier, Pool
except ImportError as exc:
    raise RuntimeError(
        "CatBoost is required. On Kaggle, use a GPU notebook image with catboost available. "
        "This notebook intentionally avoids internet installs."
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

# Required interface knobs.
FULL_RUN = True
USE_GPU = True
PRIMARY_POSITIVE_RATE = 0.20
RATE_GRID = [0.18, 0.20, 0.22, 0.24]
NEGATIVE_SAMPLE_SCALE = 1.0
PREDICT_CHUNK_SIZE = 150_000

# Reproducibility and resource controls.
SEED = 42
VALID_SIZE = 0.15
FEATURE_CHUNK_SIZE = 150_000
NEGATIVE_SCORE_CHUNK_SIZE = 200_000
SMOKE_SUBMISSION_ROW_LIMIT = 400_000
SMOKE_POSITIVE_LIMIT = 30_000

# Vectorizer controls. These are intentionally local TF-IDF features, not downloaded embeddings.
WORD_MAX_FEATURES = 220_000
CHAR_MAX_FEATURES = 180_000
WORD_MIN_DF = 2
CHAR_MIN_DF = 3
FULL_WORD_WEIGHT = 0.65

# Around 950k negatives at scale=1.0, mixed toward easy negatives with a smaller mid/hard band.
NEGATIVE_TARGETS_FULL = {"low": 600_000, "mid": 250_000, "hard": 100_000}
NEGATIVE_TARGETS_SMOKE = {"low": 35_000, "mid": 12_000, "hard": 3_000}

CATBOOST_ITERATIONS = 1200 if FULL_RUN else 250
CATBOOST_LEARNING_RATE = 0.045 if FULL_RUN else 0.06
CATBOOST_DEPTH = 8
CATBOOST_L2 = 4.0
CATBOOST_RANDOM_STRENGTH = 0.4
CATBOOST_SUBSAMPLE = 0.88
CATBOOST_VERBOSE = 100
CATBOOST_DEVICES = "0"

KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path("../outputs/vector_space_negatives_catboost_kaggle")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORES_PATH = OUTPUT_DIR / "vector_neg_catboost_scores.csv"
MODEL_PATH = OUTPUT_DIR / "vector_neg_catboost_model.cbm"

print(f"FULL_RUN={FULL_RUN} USE_GPU={USE_GPU} OUTPUT_DIR={OUTPUT_DIR.resolve()}")


In [ ]:
REQUIRED_DATA_FILES = {
    "items.csv",
    "terms.csv",
    "training_pairs.csv",
    "submission_pairs.csv",
    "sample_submission.csv",
}

EXPECTED_COLUMNS = {
    "items.csv": {"item_id", "title", "category", "brand", "gender", "age_group", "attributes"},
    "terms.csv": {"term_id", "query"},
    "training_pairs.csv": {"term_id", "item_id"},
    "submission_pairs.csv": {"id", "term_id", "item_id"},
    "sample_submission.csv": {"id", "prediction"},
}


def has_required_files(path: Path) -> bool:
    return path.is_dir() and all((path / name).exists() for name in REQUIRED_DATA_FILES)


def discover_data_dir() -> Path:
    candidates: list[Path] = []
    direct_candidates = [
        Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle"),
        Path("/kaggle/input"),
        Path("/kaggle/working/data"),
        Path("../data"),
        Path("data"),
        Path.cwd() / "data",
    ]

    for candidate in direct_candidates:
        if has_required_files(candidate):
            candidates.append(candidate)

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for item_file in kaggle_input.rglob("items.csv"):
            parent = item_file.parent
            if has_required_files(parent):
                candidates.append(parent)

    unique: list[Path] = []
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(candidate)

    if not unique:
        searched = [str(path) for path in direct_candidates]
        raise FileNotFoundError(f"Could not find data directory with {sorted(REQUIRED_DATA_FILES)}. Searched: {searched}")

    print("Data directory candidates:")
    for candidate in unique:
        print(f"  - {candidate.resolve()}")
    return unique[0]


def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        next(reader, None)
        return sum(1 for _ in reader)


def check_csv_header(path: Path, expected: set[str]) -> None:
    actual = set(pd.read_csv(path, nrows=0).columns)
    missing = expected - actual
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}; actual={sorted(actual)}")


DATA_DIR = discover_data_dir()
for filename, columns in EXPECTED_COLUMNS.items():
    check_csv_header(DATA_DIR / filename, columns)

file_audit = pd.DataFrame(
    [
        {
            "file": filename,
            "rows": count_csv_rows(DATA_DIR / filename),
            "size_mb": round((DATA_DIR / filename).stat().st_size / 1024 / 1024, 2),
        }
        for filename in sorted(REQUIRED_DATA_FILES)
    ]
)
display(file_audit)


In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
TEXT_SPLIT_RE = re.compile(r"[^0-9a-z]+")

ASCII_TRANSLATION = str.maketrans(
    {
        "\u00e7": "c", "\u011f": "g", "\u0131": "i", "\u00f6": "o", "\u015f": "s", "\u00fc": "u",
        "\u00e2": "a", "\u00ee": "i", "\u00fb": "u",
        "\u00c7": "c", "\u011e": "g", "\u0130": "i", "I": "i", "\u00d6": "o", "\u015e": "s", "\u00dc": "u",
        "\u00c2": "a", "\u00ce": "i", "\u00db": "u",
    }
)

STOPWORDS = {
    "a", "ve", "ile", "icin", "bir", "adet", "ad", "li", "set", "seti", "takim", "takimi",
    "urun", "model", "uyumlu", "orjinal", "orijinal", "ozel", "yeni",
}

COLORS = {
    "beyaz", "siyah", "kirmizi", "mavi", "yesil", "sari", "pembe", "mor", "gri", "lacivert",
    "kahverengi", "bej", "krem", "turuncu", "bordo", "gold", "altin", "gumus", "haki", "taba",
    "lila", "ekru", "vizon", "fume", "fueme", "antrasit",
}

GENDER_ALLOWED = {
    "kadin": {"kadin", "unisex", "unknown"},
    "bayan": {"kadin", "unisex", "unknown"},
    "erkek": {"erkek", "unisex", "unknown"},
    "unisex": {"kadin", "erkek", "unisex", "unknown"},
}

AGE_ALLOWED = {
    "bebek": {"bebek", "unknown"},
    "cocuk": {"cocuk", "bebek", "unknown"},
    "kiz": {"cocuk", "bebek", "unknown"},
    "oglan": {"cocuk", "bebek", "unknown"},
}


def normalize_text(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).translate(ASCII_TRANSLATION).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " ").replace("+", " ")
    text = TEXT_SPLIT_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def stem_token(token: str) -> str:
    if len(token) <= 4:
        return token
    for suffix in (
        "larin", "lerin", "lari", "leri", "lara", "lere", "larda", "lerde", "ndan", "nden",
        "dan", "den", "nin", "nun", "ler", "lar", "lik", "luk", "cuk", "cik", "ci", "cu", "si", "su", "u", "i",
    ):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token


@lru_cache(maxsize=1_000_000)
def token_set(text: str) -> frozenset[str]:
    out: set[str] = set()
    for token in TOKEN_RE.findall(text or ""):
        if token in STOPWORDS:
            continue
        out.add(token)
        stemmed = stem_token(token)
        if stemmed not in STOPWORDS:
            out.add(stemmed)
    return frozenset(out)


def query_tokens(text: str) -> tuple[str, ...]:
    seen: set[str] = set()
    out: list[str] = []
    for token in TOKEN_RE.findall(text or ""):
        if token in STOPWORDS:
            continue
        for candidate in (token, stem_token(token)):
            if candidate and candidate not in STOPWORDS and candidate not in seen:
                seen.add(candidate)
                out.append(candidate)
                break
    return tuple(out)


def coverage(query: tuple[str, ...], field: frozenset[str]) -> float:
    if not query:
        return 0.0
    return sum(1 for token in query if token in field) / len(query)


def split_category(value: object) -> tuple[str, str, str, str, int]:
    parts = [normalize_text(part) for part in str(value or "").split("/")]
    parts = [part for part in parts if part]
    if not parts:
        parts = ["unknown"]
    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf, len(parts)


def weighted_item_text(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["title_norm"] + " " + frame["title_norm"] + " " + frame["title_norm"] + " " +
        frame["category_norm"] + " " + frame["category_norm"] + " " +
        frame["brand_norm"] + " " + frame["brand_norm"] + " " +
        frame["attributes_norm"]
    ).str.strip()


In [ ]:
start = time.time()

terms = pd.read_csv(DATA_DIR / "terms.csv", dtype=str, keep_default_na=False)
items = pd.read_csv(DATA_DIR / "items.csv", dtype=str, keep_default_na=False)
training_pairs = pd.read_csv(DATA_DIR / "training_pairs.csv", dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str, "prediction": "int8"}, keep_default_na=False)

for column in ["term_id", "query"]:
    if column not in terms.columns:
        raise ValueError(f"terms.csv missing {column}")
for column in ["item_id", "title", "category", "brand", "gender", "age_group", "attributes"]:
    if column not in items.columns:
        raise ValueError(f"items.csv missing {column}")

terms["query_norm"] = terms["query"].map(normalize_text)
terms["query_tokens"] = terms["query_norm"].map(query_tokens)
terms["query_colors"] = terms["query_tokens"].map(lambda tokens: tuple(token for token in tokens if token in COLORS))

for column in ["title", "category", "brand", "gender", "age_group", "attributes"]:
    items[f"{column}_norm"] = items[column].map(normalize_text)

category_levels = pd.DataFrame(
    items["category"].map(split_category).tolist(),
    columns=["item_cat_l1", "item_cat_l2", "item_cat_l3", "item_cat_leaf", "category_depth"],
    index=items.index,
)
items = pd.concat([items, category_levels], axis=1)
items["item_brand"] = items["brand_norm"].where(items["brand_norm"].ne(""), "unknown")
items["item_gender"] = items["gender_norm"].where(items["gender_norm"].ne(""), "unknown")
items["item_age_group"] = items["age_group_norm"].where(items["age_group_norm"].ne(""), "unknown")
items["item_text_weighted"] = weighted_item_text(items)

term_index = pd.Index(terms["term_id"].astype(str), name="term_id")
item_index = pd.Index(items["item_id"].astype(str), name="item_id")
if not term_index.is_unique:
    raise ValueError("terms.term_id must be unique")
if not item_index.is_unique:
    raise ValueError("items.item_id must be unique")

TERM_QUERY = terms["query_norm"].to_numpy(dtype=object)
TERM_TOKENS = terms["query_tokens"].to_numpy(dtype=object)
TERM_COLORS = terms["query_colors"].to_numpy(dtype=object)

ITEM_TITLE = items["title_norm"].to_numpy(dtype=object)
ITEM_CATEGORY = items["category_norm"].to_numpy(dtype=object)
ITEM_BRAND = items["brand_norm"].to_numpy(dtype=object)
ITEM_ATTR = items["attributes_norm"].to_numpy(dtype=object)
ITEM_FULL_TEXT = items["item_text_weighted"].to_numpy(dtype=object)
ITEM_CAT_L1 = items["item_cat_l1"].to_numpy(dtype=object)
ITEM_CAT_L2 = items["item_cat_l2"].to_numpy(dtype=object)
ITEM_CAT_L3 = items["item_cat_l3"].to_numpy(dtype=object)
ITEM_BRAND_CAT = items["item_brand"].to_numpy(dtype=object)
ITEM_GENDER = items["item_gender"].to_numpy(dtype=object)
ITEM_AGE_GROUP = items["item_age_group"].to_numpy(dtype=object)
ITEM_CATEGORY_DEPTH = items["category_depth"].to_numpy(dtype=np.float32)

blank_audit = pd.DataFrame(
    [
        {"table": "terms", "column": col, "blank_count": int(terms[col].astype(str).eq("").sum())}
        for col in ["term_id", "query", "query_norm"]
    ]
    + [
        {"table": "items", "column": col, "blank_count": int(items[col].astype(str).eq("").sum())}
        for col in ["item_id", "title", "category", "brand", "gender", "age_group", "attributes"]
    ]
    + [
        {"table": "training_pairs", "column": col, "blank_count": int(training_pairs[col].astype(str).eq("").sum())}
        for col in ["term_id", "item_id"]
    ]
)

display(blank_audit)
print(f"terms={len(terms):,} items={len(items):,} training_pairs={len(training_pairs):,} sample_submission={len(sample_submission):,}")
print(f"loaded and normalized in {(time.time() - start) / 60:.2f} min")


In [ ]:
start = time.time()

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=WORD_MIN_DF,
    max_features=WORD_MAX_FEATURES,
    dtype=np.float32,
    norm="l2",
    sublinear_tf=True,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=CHAR_MIN_DF,
    max_features=CHAR_MAX_FEATURES,
    dtype=np.float32,
    norm="l2",
    sublinear_tf=True,
)

fit_corpus = chain(terms["query_norm"].astype(str), items["item_text_weighted"].astype(str))
word_vectorizer.fit(fit_corpus)
fit_corpus = chain(terms["query_norm"].astype(str), items["item_text_weighted"].astype(str))
char_vectorizer.fit(fit_corpus)

term_word = word_vectorizer.transform(terms["query_norm"].astype(str))
item_full_word = word_vectorizer.transform(items["item_text_weighted"].astype(str))
term_char = char_vectorizer.transform(terms["query_norm"].astype(str))
item_full_char = char_vectorizer.transform(items["item_text_weighted"].astype(str))

gc.collect()
print(f"word vocab={len(word_vectorizer.vocabulary_):,} char vocab={len(char_vectorizer.vocabulary_):,}")
print(f"term_word={term_word.shape} item_full_word={item_full_word.shape}")
print(f"term_char={term_char.shape} item_full_char={item_full_char.shape}")
print(f"vectorizers fitted/transformed in {(time.time() - start) / 60:.2f} min")


In [ ]:
VECTOR_FEATURES = [
    "word_sim_full",
    "char_sim_full",
    "word_sim_title",
    "word_sim_category",
    "word_sim_brand",
    "word_sim_attributes",
    "char_sim_title",
    "char_sim_attributes",
]

LEXICAL_BASE_FEATURES = [
    "n_query_tokens",
    "query_len_chars",
    "title_len_tokens",
    "category_len_tokens",
    "brand_len_tokens",
    "attr_len_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "max_field_cov",
    "shared_title_tokens",
    "shared_category_tokens",
    "shared_brand_tokens",
    "shared_attr_tokens",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "category_depth",
]

LEXICAL_FEATURES = LEXICAL_BASE_FEATURES + ["lexical_score"]
CAT_FEATURES = ["item_cat_l1", "item_cat_l2", "item_cat_l3", "item_brand", "item_gender", "item_age_group"]
FEATURE_NAMES = VECTOR_FEATURES + LEXICAL_FEATURES + CAT_FEATURES


def cosine_rows(left: sparse.spmatrix, right: sparse.spmatrix) -> np.ndarray:
    values = left.multiply(right).sum(axis=1)
    return np.asarray(values).ravel().astype(np.float32)


def get_row_positions(rows: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    term_pos = term_index.get_indexer(rows["term_id"].astype(str).to_numpy())
    item_pos = item_index.get_indexer(rows["item_id"].astype(str).to_numpy())
    if (term_pos < 0).any():
        missing = rows.loc[term_pos < 0, "term_id"].head(5).tolist()
        raise KeyError(f"Unknown term_id values, examples={missing}")
    if (item_pos < 0).any():
        missing = rows.loc[item_pos < 0, "item_id"].head(5).tolist()
        raise KeyError(f"Unknown item_id values, examples={missing}")
    return term_pos, item_pos


def score_rows_full_similarity(rows: pd.DataFrame) -> np.ndarray:
    if len(rows) == 0:
        return np.empty(0, dtype=np.float32)
    term_pos, item_pos = get_row_positions(rows)
    word_score = cosine_rows(term_word[term_pos], item_full_word[item_pos])
    char_score = cosine_rows(term_char[term_pos], item_full_char[item_pos])
    return (FULL_WORD_WEIGHT * word_score + (1.0 - FULL_WORD_WEIGHT) * char_score).astype(np.float32)


def score_rows_full_similarity_chunked(rows: pd.DataFrame, chunk_size: int = 250_000) -> np.ndarray:
    parts: list[np.ndarray] = []
    for start_idx in tqdm(range(0, len(rows), chunk_size), desc="full similarity chunks"):
        parts.append(score_rows_full_similarity(rows.iloc[start_idx : start_idx + chunk_size]))
    return np.concatenate(parts) if parts else np.empty(0, dtype=np.float32)


def pair_keys(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def rows_to_features(rows: pd.DataFrame, desc: str = "features") -> pd.DataFrame:
    rows = rows[["term_id", "item_id"]].reset_index(drop=True)
    n_rows = len(rows)
    term_pos, item_pos = get_row_positions(rows)

    frame = pd.DataFrame(index=np.arange(n_rows))
    q_word = term_word[term_pos]
    q_char = term_char[term_pos]

    frame["word_sim_full"] = cosine_rows(q_word, item_full_word[item_pos])
    frame["char_sim_full"] = cosine_rows(q_char, item_full_char[item_pos])

    field_texts = {
        "title": ITEM_TITLE[item_pos],
        "category": ITEM_CATEGORY[item_pos],
        "brand": ITEM_BRAND[item_pos],
        "attributes": ITEM_ATTR[item_pos],
    }
    for field_name, texts in field_texts.items():
        field_matrix = word_vectorizer.transform(texts)
        frame[f"word_sim_{field_name}"] = cosine_rows(q_word, field_matrix)
        del field_matrix

    for field_name in ["title", "attributes"]:
        field_matrix = char_vectorizer.transform(field_texts[field_name])
        frame[f"char_sim_{field_name}"] = cosine_rows(q_char, field_matrix)
        del field_matrix

    del q_word, q_char

    lexical = np.zeros((n_rows, len(LEXICAL_BASE_FEATURES)), dtype=np.float32)
    for idx in tqdm(range(n_rows), desc=f"{desc} lexical", leave=False):
        q_tokens = TERM_TOKENS[term_pos[idx]]
        q_colors = set(TERM_COLORS[term_pos[idx]])
        query = TERM_QUERY[term_pos[idx]]
        title = ITEM_TITLE[item_pos[idx]]
        category = ITEM_CATEGORY[item_pos[idx]]
        brand = ITEM_BRAND[item_pos[idx]]
        attrs = ITEM_ATTR[item_pos[idx]]
        gender = ITEM_GENDER[item_pos[idx]] or "unknown"
        age_group = ITEM_AGE_GROUP[item_pos[idx]] or "unknown"

        title_tokens = token_set(title)
        category_tokens = token_set(category)
        brand_tokens = token_set(brand)
        attr_tokens = token_set(attrs)
        full_tokens = title_tokens | category_tokens | brand_tokens | attr_tokens

        title_cov = coverage(q_tokens, title_tokens)
        category_cov = coverage(q_tokens, category_tokens)
        brand_cov = coverage(q_tokens, brand_tokens)
        attr_cov = coverage(q_tokens, attr_tokens)
        full_cov = coverage(q_tokens, full_tokens)
        max_field_cov = max(title_cov, category_cov, brand_cov, attr_cov)

        gender_mismatch = 0.0
        for gender_word, allowed in GENDER_ALLOWED.items():
            if gender_word in q_tokens and gender not in allowed and gender_word not in full_tokens:
                gender_mismatch = 1.0
                break

        age_mismatch = 0.0
        for age_word, allowed in AGE_ALLOWED.items():
            if age_word in q_tokens and age_group not in allowed and age_word not in full_tokens:
                age_mismatch = 1.0
                break

        color_mismatch = float(bool(q_colors and full_tokens.isdisjoint(q_colors)))
        weak_match = float(full_cov < 0.34 and max_field_cov < 0.40)

        values = {
            "n_query_tokens": float(len(q_tokens)),
            "query_len_chars": float(len(query)),
            "title_len_tokens": float(len(title_tokens)),
            "category_len_tokens": float(len(category_tokens)),
            "brand_len_tokens": float(len(brand_tokens)),
            "attr_len_tokens": float(len(attr_tokens)),
            "title_cov": title_cov,
            "category_cov": category_cov,
            "brand_cov": brand_cov,
            "attr_cov": attr_cov,
            "full_cov": full_cov,
            "max_field_cov": max_field_cov,
            "shared_title_tokens": float(sum(1 for token in q_tokens if token in title_tokens)),
            "shared_category_tokens": float(sum(1 for token in q_tokens if token in category_tokens)),
            "shared_brand_tokens": float(sum(1 for token in q_tokens if token in brand_tokens)),
            "shared_attr_tokens": float(sum(1 for token in q_tokens if token in attr_tokens)),
            "query_in_title": float(bool(query and query in title)),
            "query_in_category": float(bool(query and query in category)),
            "query_in_brand": float(bool(query and query in brand)),
            "full_complete": float(full_cov >= 0.999),
            "title_complete": float(title_cov >= 0.999),
            "category_complete": float(category_cov >= 0.999),
            "brand_complete": float(brand_cov >= 0.999 and bool(brand)),
            "short_complete": float(len(q_tokens) <= 2 and full_cov >= 0.999),
            "gender_mismatch": gender_mismatch,
            "age_mismatch": age_mismatch,
            "color_mismatch": color_mismatch,
            "weak_match": weak_match,
            "category_depth": ITEM_CATEGORY_DEPTH[item_pos[idx]],
        }
        lexical[idx] = [values[name] for name in LEXICAL_BASE_FEATURES]

    lexical_frame = pd.DataFrame(lexical, columns=LEXICAL_BASE_FEATURES)
    frame = pd.concat([frame, lexical_frame], axis=1)

    frame["lexical_score"] = (
        1.75 * frame["word_sim_full"]
        + 1.20 * frame["char_sim_full"]
        + 0.95 * frame["word_sim_title"]
        + 0.55 * frame["word_sim_category"]
        + 0.65 * frame["word_sim_brand"]
        + 0.35 * frame["word_sim_attributes"]
        + 2.05 * frame["full_cov"]
        + 1.35 * frame["title_cov"]
        + 0.90 * frame["category_cov"]
        + 0.95 * frame["brand_cov"]
        + 0.45 * frame["attr_cov"]
        + 0.80 * frame["query_in_title"]
        + 0.65 * frame["query_in_category"]
        + 0.85 * frame["query_in_brand"]
        + 0.95 * frame["full_complete"]
        + 0.65 * frame["title_complete"]
        + 0.50 * frame["category_complete"]
        + 0.55 * frame["brand_complete"]
        + 0.40 * frame["short_complete"]
        - 0.80 * frame["gender_mismatch"]
        - 0.70 * frame["age_mismatch"]
        - 0.60 * frame["color_mismatch"]
        - 0.85 * frame["weak_match"]
    ).astype(np.float32)

    frame["item_cat_l1"] = ITEM_CAT_L1[item_pos]
    frame["item_cat_l2"] = ITEM_CAT_L2[item_pos]
    frame["item_cat_l3"] = ITEM_CAT_L3[item_pos]
    frame["item_brand"] = ITEM_BRAND_CAT[item_pos]
    frame["item_gender"] = ITEM_GENDER[item_pos]
    frame["item_age_group"] = ITEM_AGE_GROUP[item_pos]

    for name in CAT_FEATURES:
        frame[name] = frame[name].astype(str).replace({"": "unknown", "nan": "unknown"})

    return frame[FEATURE_NAMES]


def rows_to_features_chunked(rows: pd.DataFrame, chunk_size: int = FEATURE_CHUNK_SIZE, desc: str = "features") -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for start_idx in tqdm(range(0, len(rows), chunk_size), desc=f"{desc} chunks"):
        part = rows_to_features(rows.iloc[start_idx : start_idx + chunk_size], desc=f"{desc} {start_idx:,}")
        parts.append(part)
        gc.collect()
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=FEATURE_NAMES)

print(f"feature_count={len(FEATURE_NAMES)} cat_features={CAT_FEATURES}")


In [ ]:
if "label" in training_pairs.columns:
    positive_mask = training_pairs["label"].astype(str).ne("0")
else:
    positive_mask = np.ones(len(training_pairs), dtype=bool)

positive_pairs_all = training_pairs.loc[positive_mask, ["term_id", "item_id"]].reset_index(drop=True)
if len(positive_pairs_all) == 0:
    raise ValueError("No positive rows found in training_pairs.csv")

positive_key_set = set(pair_keys(positive_pairs_all))
print(f"positive_pairs={len(positive_pairs_all):,} unique_positive_pairs={len(positive_key_set):,}")

positive_scores = score_rows_full_similarity_chunked(positive_pairs_all, chunk_size=250_000)
positive_score_summary = pd.Series(positive_scores, name="positive_vec_similarity").describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
print(positive_score_summary.to_string())

pos_q05, pos_q10, pos_q25, pos_q50 = np.quantile(positive_scores, [0.05, 0.10, 0.25, 0.50])
LOW_MAX = max(0.005, float(pos_q10 * 0.60))
MID_MAX = max(LOW_MAX + 0.01, float(pos_q25 * 0.80))
HARD_MAX = max(MID_MAX + 0.01, float(pos_q50 * 0.90))

print(f"negative bands: low <= {LOW_MAX:.5f}; mid <= {MID_MAX:.5f}; hard <= {HARD_MAX:.5f}")


In [ ]:
def sample_frame(frame: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(frame) <= n:
        return frame.copy()
    return frame.sample(n=n, random_state=seed)


def collect_negative_pools(row_limit: int | None = None) -> dict[str, pd.DataFrame]:
    pools: dict[str, list[pd.DataFrame]] = {"low": [], "mid": [], "hard": []}
    processed = 0
    reader = pd.read_csv(
        DATA_DIR / "submission_pairs.csv",
        usecols=["id", "term_id", "item_id"],
        dtype=str,
        keep_default_na=False,
        chunksize=NEGATIVE_SCORE_CHUNK_SIZE,
    )

    for chunk in tqdm(reader, desc="score submission pairs for negatives"):
        if row_limit is not None:
            remaining = row_limit - processed
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)

        raw_chunk_len = len(chunk)
        scores = score_rows_full_similarity(chunk)
        chunk = chunk.assign(vec_similarity=scores)
        keep = ~pair_keys(chunk).isin(positive_key_set)
        chunk = chunk.loc[keep]

        low_mask = chunk["vec_similarity"].le(LOW_MAX)
        mid_mask = chunk["vec_similarity"].gt(LOW_MAX) & chunk["vec_similarity"].le(MID_MAX)
        hard_mask = chunk["vec_similarity"].gt(MID_MAX) & chunk["vec_similarity"].le(HARD_MAX)

        for name, mask in [("low", low_mask), ("mid", mid_mask), ("hard", hard_mask)]:
            if mask.any():
                pools[name].append(chunk.loc[mask, ["id", "term_id", "item_id", "vec_similarity"]].copy())

        processed += raw_chunk_len
        del chunk, scores
        gc.collect()

    out: dict[str, pd.DataFrame] = {}
    for name, parts in pools.items():
        out[name] = pd.concat(parts, ignore_index=True).drop_duplicates("id") if parts else pd.DataFrame(columns=["id", "term_id", "item_id", "vec_similarity"])
        print(f"{name}_pool={len(out[name]):,}")
    return out


def build_vector_negatives() -> pd.DataFrame:
    base_targets = NEGATIVE_TARGETS_FULL if FULL_RUN else NEGATIVE_TARGETS_SMOKE
    targets = {name: max(0, int(round(count * NEGATIVE_SAMPLE_SCALE))) for name, count in base_targets.items()}
    row_limit = None if FULL_RUN else SMOKE_SUBMISSION_ROW_LIMIT
    print(f"negative targets={targets} row_limit={row_limit}")

    pools = collect_negative_pools(row_limit=row_limit)
    sampled_parts: list[pd.DataFrame] = []
    for offset, name in enumerate(["low", "mid", "hard"]):
        target = targets[name]
        pool = pools[name]
        if target <= 0 or len(pool) == 0:
            continue
        sampled = sample_frame(pool, target, SEED + 10 + offset)
        sampled["negative_band"] = name
        sampled_parts.append(sampled)
        print(f"sampled {name}: {len(sampled):,} from {len(pool):,}")

    negatives = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else pd.DataFrame(columns=["id", "term_id", "item_id", "vec_similarity", "negative_band"])
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    overlap = pair_keys(negatives).isin(positive_key_set).sum()
    if overlap:
        raise AssertionError(f"Generated negatives overlap positives: {overlap}")

    print("negative vector similarity distribution")
    print(negatives["vec_similarity"].describe(percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]).to_string())
    display(negatives.groupby("negative_band")["vec_similarity"].agg(["count", "min", "mean", "max"]))
    return negatives

negatives = build_vector_negatives()
print(f"final_negatives={len(negatives):,}")


In [ ]:
if FULL_RUN:
    positive_train = positive_pairs_all.copy()
else:
    positive_train = positive_pairs_all.sample(n=min(SMOKE_POSITIVE_LIMIT, len(positive_pairs_all)), random_state=SEED).reset_index(drop=True)

positive_train = positive_train[["term_id", "item_id"]].copy()
positive_train["label"] = 1
negative_train = negatives[["term_id", "item_id"]].copy()
negative_train["label"] = 0

train_rows = pd.concat([positive_train, negative_train], ignore_index=True)
labels = train_rows["label"].to_numpy(dtype=np.int8)

assert set(np.unique(labels)) == {0, 1}, "Synthetic training labels must contain positives and negatives"
assert int(labels.sum()) == len(positive_train), "Positive label count mismatch"
assert pair_keys(negative_train).isin(positive_key_set).sum() == 0, "Generated negatives must not include known positives"
if FULL_RUN:
    assert len(positive_train) == len(positive_pairs_all), "FULL_RUN should keep every original positive pair"

print(f"training_rows={len(train_rows):,} positives={int(labels.sum()):,} negatives={int((labels == 0).sum()):,} positive_rate={labels.mean():.4f}")
display(pd.Series(labels, name="label").value_counts().rename_axis("label").to_frame("count"))

start = time.time()
x = rows_to_features_chunked(train_rows, chunk_size=FEATURE_CHUNK_SIZE, desc="train")
print(f"x_shape={x.shape} built in {(time.time() - start) / 60:.2f} min")
print(x.dtypes.tail(10))


In [ ]:
x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    labels,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=labels,
)

del x, train_rows
_ = gc.collect()

params = {
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": CATBOOST_ITERATIONS,
    "learning_rate": CATBOOST_LEARNING_RATE,
    "depth": CATBOOST_DEPTH,
    "l2_leaf_reg": CATBOOST_L2,
    "random_strength": CATBOOST_RANDOM_STRENGTH,
    "bootstrap_type": "Bernoulli",
    "subsample": CATBOOST_SUBSAMPLE,
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "allow_writing_files": False,
    "verbose": CATBOOST_VERBOSE,
}
if USE_GPU:
    params.update({"task_type": "GPU", "devices": CATBOOST_DEVICES, "gpu_ram_part": 0.90})
else:
    params.update({"thread_count": -1})

train_pool = Pool(x_train, y_train, cat_features=CAT_FEATURES)
valid_pool = Pool(x_valid, y_valid, cat_features=CAT_FEATURES)

model = CatBoostClassifier(**params)
try:
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=120)
except Exception as exc:
    if not USE_GPU:
        raise
    warnings.warn(f"GPU CatBoost failed ({exc}). Retrying on CPU.")
    params.pop("task_type", None)
    params.pop("devices", None)
    params.pop("gpu_ram_part", None)
    params["thread_count"] = -1
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=120)

valid_prob = model.predict_proba(valid_pool)[:, 1]
best_score = -1.0
best_threshold = 0.5
for threshold in np.linspace(0.05, 0.95, 91):
    pred = (valid_prob >= threshold).astype(np.int8)
    score = f1_score(y_valid, pred, average="macro")
    if score > best_score:
        best_score = float(score)
        best_threshold = float(threshold)

print(f"synthetic valid macro_f1={best_score:.5f} threshold={best_threshold:.3f}")
print(f"valid probability summary:\n{pd.Series(valid_prob).describe(percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]).to_string()}")

feature_importance = pd.Series(model.get_feature_importance(valid_pool), index=FEATURE_NAMES).sort_values(ascending=False)
display(feature_importance.head(40).to_frame("importance"))

model.save_model(str(MODEL_PATH))
print(f"saved model: {MODEL_PATH}")


In [ ]:
def predict_pairs_to_file(model: CatBoostClassifier, output_path: Path, row_limit: int | None = None) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(
            DATA_DIR / "submission_pairs.csv",
            usecols=["id", "term_id", "item_id"],
            dtype=str,
            keep_default_na=False,
            chunksize=PREDICT_CHUNK_SIZE,
        )
        for chunk in tqdm(reader, desc="predict submission pairs"):
            if row_limit is not None:
                remaining = row_limit - written
                if remaining <= 0:
                    break
                chunk = chunk.head(remaining)

            matrix = rows_to_features(chunk, desc=f"predict {written:,}")
            pool = Pool(matrix, cat_features=CAT_FEATURES)
            prob = model.predict_proba(pool)[:, 1]
            writer.writerows((row_id, term_id, f"{score:.8f}") for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob))
            written += len(chunk)
            del matrix, pool, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")
    return output_path

prediction_row_limit = None if FULL_RUN else SMOKE_SUBMISSION_ROW_LIMIT
predict_pairs_to_file(model, SCORES_PATH, row_limit=prediction_row_limit)

scores_preview = pd.read_csv(SCORES_PATH, usecols=["id", "prob"])
if FULL_RUN:
    assert len(scores_preview) == len(sample_submission), "Score row count must match sample_submission"
    assert scores_preview["id"].astype(str).equals(sample_submission["id"].astype(str)), "Score id order must match sample_submission"
print(scores_preview["prob"].describe(percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]).to_string())


In [ ]:
def rate_suffix(rate: float) -> str:
    return f"r{int(round(rate * 100)):02d}"


def write_rate_submission(scores: pd.DataFrame, rate: float) -> Path:
    threshold = float(scores["prob"].quantile(1.0 - rate))
    prediction = (scores["prob"].to_numpy() >= threshold).astype(np.int8)
    output = pd.DataFrame({"id": scores["id"].astype(str), "prediction": prediction})

    if len(output) == len(sample_submission):
        assert output["id"].equals(sample_submission["id"].astype(str)), "Submission id order must match sample_submission"

    output_path = OUTPUT_DIR / f"submission_vector_neg_catboost_{rate_suffix(rate)}.csv"
    output.to_csv(output_path, index=False)
    print(f"{output_path.name}: threshold={threshold:.8f} rows={len(output):,} positives={int(prediction.sum()):,} rate={prediction.mean():.5f}")
    return output_path

scores = pd.read_csv(SCORES_PATH, dtype={"id": str, "term_id": str, "prob": np.float32})
if FULL_RUN:
    assert len(scores) == len(sample_submission), "Scores must cover every sample submission row"
    assert scores["id"].equals(sample_submission["id"].astype(str)), "Scores must preserve sample submission id order"

submission_paths = []
for rate in sorted(set(RATE_GRID + [PRIMARY_POSITIVE_RATE])):
    submission_paths.append(write_rate_submission(scores, rate))

primary_path = OUTPUT_DIR / f"submission_vector_neg_catboost_{rate_suffix(PRIMARY_POSITIVE_RATE)}.csv"
print(f"primary submission: {primary_path}")


In [ ]:
output_audit = []
for path in [SCORES_PATH, MODEL_PATH] + submission_paths:
    if path.exists():
        output_audit.append({"path": str(path), "size_mb": round(path.stat().st_size / 1024 / 1024, 2)})
display(pd.DataFrame(output_audit))
print("Done.")
